# Module 6: Govern model and tool calls
> ~30 min · optional

In this module, policies you set in LangSmith govern the agent without changing its code, so security and budget rules live in one place instead of in every app. The **LLM Gateway** sits between the agent and the model provider: it redacts secrets before the provider sees them and blocks a team once it spends its cap. The **Tool Gateway** serves the agent's web search from a **tool kit** and can block a tool call by policy. Every decision is recorded in a trace.

## Setup
This module uses two keys from your `.env`, and each one decides which organization a gateway acts in:

| Sections | Gateway | Key | Organization |
|---|---|---|---|
| 1–3 | LLM Gateway | `LANGSMITH_GATEWAY_API_KEY` | the one that owns that key, which may not be yours |
| 4–5 | Tool Gateway | `LANGSMITH_API_KEY` (a personal access token, `lsv2_pt_`) | this workspace's |

So the LLM Gateway policies, traces and spend in sections 1–3 live in the gateway key's organization, not in the workspace you trace to. The cell below prints both keys' types and workspaces.

In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.config import environment_project  # sets the tracing destination before any LangSmith client exists
from langsmith import Client, uuid7

from utils.gateway import describe_key, gateway_workspace_id
from utils.notebook import trace_link

client = Client()
dest = environment_project(client)
gateway_workspace = gateway_workspace_id()  # policies below are scoped to this workspace


def trace_url(run_id):
    """Return the LangSmith URL of the trace whose root run is run_id."""
    return f"{dest.url}/r/{run_id}?poll=true"


print("LLM Gateway key: ", describe_key(os.environ["LANGSMITH_GATEWAY_API_KEY"]), "· workspace", gateway_workspace)
print("Tool Gateway key:", describe_key(os.environ["LANGSMITH_API_KEY"]), "· workspace", os.environ["LANGSMITH_WORKSPACE_ID"])
trace_link(dest.url, "Agent traces (Development)")

## 1. Route model calls through the LLM Gateway
Every model call in this tour already goes through the **LLM Gateway**: `utils/models.py` points the model at `https://gateway.smith.langchain.com/openai` and authenticates with `LANGSMITH_GATEWAY_API_KEY`. The gateway calls the provider with a key stored in that key's workspace, applies that organization's policies, and returns metadata about what it did on every response. Provider keys, policies and cost tracking live in one place instead of in each app.

The first cell builds the tour agent on that gateway-routed model. The second asks one news question and counts the model calls the gateway reported on; expect an answer, a call count of usually two or more (one per agent step), and the outcome the gateway recorded for each call. The run takes about 30 seconds.

> **Requires:** LLM Gateway (beta) in the organization that owns `LANGSMITH_GATEWAY_API_KEY`, a provider key or Gateway Credits there, and `gateway:invoke` on the key (Workspace Admin has it). **Without it:** read sections 1–3 and continue at section 4.

In [ ]:
from agents.client_research_agent import build_client_research_agent
from utils.gateway import GatewayInfoCollector, model_endpoint
from utils.models import make_model

tour_model = make_model()
agent = build_client_research_agent(model=tour_model)

print("Model endpoint:", model_endpoint(tour_model))

In [ ]:
collector = GatewayInfoCollector()  # reads the gateway's metadata off each model response
question = "What did NVIDIA announce this week? Search at most once."

run_id = uuid7()  # lets the cell link straight to this trace
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"run_id": run_id, "configurable": {"thread_id": str(uuid4())}, "callbacks": [collector]},
)

print("Answer:", result["messages"][-1].text[:300])
print("Model calls through the gateway:", len(collector.infos))
print("Outcomes:", sorted({info.get("outcome", "unknown") for info in collector.infos}))
trace_link(trace_url(run_id), "Agent trace")

Each model run in your agent's trace carries what the gateway reported about that call. The gateway also keeps its own trace of every call, in **LLM Gateway** → **Traces** of the organization that owns `LANGSMITH_GATEWAY_API_KEY`, which may not be this workspace. Each call is a row with tokens, cost and `gateway.credential_source`.

> For apps that build their own `ChatAnthropic` model or run Deep Agents, `LANGSMITH_GATEWAY=true` is the no-code switch: the model reads it when it is built and sends its calls to the gateway.

## 2. Redact secrets before they reach the provider
A **data policy** scans each outbound request, including the tool results the agent sends back, and replaces each match with a `[SAFE_TO_USE:<CATEGORY>_<id>]` placeholder. The gateway restores the real values in the response, so the provider never sees them and your agent never notices. You get redaction without writing a scrubbing layer, and it covers every app that calls the gateway. This policy detects secrets plus one custom pattern, the firm's `ACCT-` account numbers. It applies only to requests that carry the header `X-Gateway-Tour: govern`, so nobody else's traffic changes. The first cell defines the policy; the second creates it.

> **Requires:** `organization:gateway:manage` in the organization that owns `LANGSMITH_GATEWAY_API_KEY` (Organization Admin or Operator there), not in your own organization. **Without it:** leave `create_policies = False`, and the policy cells print `Skipped`. Ask an admin to run sections 2–3, or read along.

In [ ]:
create_policies = False  # an Organization Admin or Operator in the gateway key's org sets this to True

redaction_policy = {
    "name": "tour-redact-secrets",
    "policy_type": "guard",
    "action": "block",  # the only stored action; guard policies redact rather than block
    "subject_matchers": [
        {"key": "workspace_id", "value": gateway_workspace},
        {"key": "tour", "value": "govern"},  # matches the X-Gateway-Tour header
    ],
    "config": {
        "version": 1,
        "detect": {"pii": False, "secrets": True, "custom": [{"pattern": "ACCT-[0-9]{8}"}]},
        "trace": {"capture_content": True},  # store request bodies in gateway traces, so you can see the placeholders
    },
}

In [ ]:
from utils.gateway import create_gateway_policy

if create_policies:
    redaction = create_gateway_policy(redaction_policy)
    print("Data policy:", redaction["name"] if redaction else "not created")
else:
    redaction = None
    print("Data policy: Skipped (admin verifies live)")

Now send a request with two secrets through a model that carries the policy's header. The prompt holds a made-up GitHub token, and it asks for the client's account number, so the agent calls `get_client_profile` and the `ACCT-` number arrives in a tool result. The count covers every data policy in the gateway key's organization, not only this one: if that organization already redacts secrets for all its traffic, you see redacted calls even when you skipped the policy cells. Expect `True` for the account number in the answer, because the gateway restores it.

In [ ]:
import re
import secrets
import string

fake_token = "ghp_" + "".join(secrets.choice(string.ascii_letters + string.digits) for _ in range(36))  # made up at runtime
leaky_prompt = (
    "Prep a one-paragraph meeting brief for client CL-0042 and include their account number. "
    f"From my call notes: they asked us to rotate the GitHub token {fake_token} we use for their reporting repo."
)

redacting_agent = build_client_research_agent(model=make_model(default_headers={"X-Gateway-Tour": "govern"}))

In [ ]:
collector = GatewayInfoCollector()
run_id = uuid7()  # lets the cell link straight to this trace
result = redacting_agent.invoke(
    {"messages": [{"role": "user", "content": leaky_prompt}]},
    config={"run_id": run_id, "configurable": {"thread_id": str(uuid4())}, "callbacks": [collector]},
)

answer = result["messages"][-1].text
redacted_calls = sum(1 for info in collector.infos if info.get("guard_redaction_applied"))

print("Model calls a data policy redacted:", redacted_calls, "of", len(collector.infos))  # any policy in the gateway org
print("Account number in the answer:", bool(re.search(r"ACCT-[0-9]{8}", answer)))
trace_link(trace_url(run_id), "Agent trace")

Open **LLM Gateway** → **Traces** and click the newest row. The request shows `[SAFE_TO_USE:...]` placeholders where the token and the account number were, and `policy.matched_rules` lists the rules that fired. Your agent's own trace still shows the real values, because the gateway restores them before your code sees the response.

> System prompts are not scanned, provider error bodies keep their placeholders, and a scanner that errors lets the request through unredacted.

<img src="../images/TODO-06-gateway-trace-redacted.png" alt="LLM Gateway trace with SAFE_TO_USE placeholders in the request" style="width: auto; max-height: 360px; border-radius: 8px;">

## 3. Cap spend per team
A **spend policy** blocks a scope's requests with `402` once it has spent its cap. A header condition gives each team its own cap under one API key: here, a cap of a hundredth of a cent per hour on requests that send `X-Gateway-Team: advisory-desk`. You get per-team budgets without issuing a key per team or tracking spend yourself. The request that crosses the cap still succeeds, and the next one is blocked, so after the policy cells, the run cell sends the same short question up to three times and stops at the first `402`. Each run takes about 20 seconds.

> **Requires:** `organization:gateway:manage`, as in section 2. **Without it:** the cap cell prints `Skipped` and the runs below are all allowed. Read on for what a block looks like.

In [ ]:
spend_policy = {
    "name": "tour-advisory-desk-cap",
    "policy_type": "spend_cap",
    "action": "block",
    "subject_matchers": [
        {"key": "workspace_id", "value": gateway_workspace},
        {"key": "team", "value": "advisory-desk"},  # matches the X-Gateway-Team header
    ],
    "config": {"window": "hourly", "limit_usd": 0.0001},  # low enough that one run crosses it
}

In [ ]:
if create_policies:
    spend_cap = create_gateway_policy(spend_policy)
    print("Spend cap:", f"{spend_cap['name']} ($0.0001/hour)" if spend_cap else "not created")
else:
    spend_cap = None
    print("Spend cap: Skipped (admin verifies live)")

In [ ]:
import openai

desk_agent = build_client_research_agent(model=make_model(default_headers={"X-Gateway-Team": "advisory-desk"}))
desk_question = "In two sentences, what is a money market fund? Do not search."  # no search keeps each run cheap and fast

In [ ]:
for attempt in range(1, 4):
    try:
        desk_agent.invoke(
            {"messages": [{"role": "user", "content": desk_question}]},
            config={"configurable": {"thread_id": str(uuid4())}},
        )
        print(f"Run {attempt}: allowed")
    except openai.APIStatusError as exc:
        if exc.status_code != 402:  # only the cap's block is expected here
            raise
        print(f"Run {attempt}: blocked ({exc.status_code}) {exc.message}")
        break

In the agent's trace, the blocked run ends in a model run with a `402` error naming `tour-advisory-desk-cap`. In **LLM Gateway** → **Cost Controls**, the cap's **Usage** column is over 100%. Gateway policy violations also become issues in **Engine**, the same place Module 3's issue came from.

<img src="../images/TODO-06-cost-controls-usage.png" alt="Cost Controls table with the advisory-desk cap over 100% usage" style="width: auto; max-height: 320px; border-radius: 8px;">

## 4. Serve the agent's search from a tool kit
The **Tool Gateway** exposes a **tool kit**, a set of tools picked from LangSmith built-ins and your own MCP servers, as one MCP endpoint, and it holds the credentials, so the agent's code carries none and any MCP client can use the same tools. From here on the module uses `LANGSMITH_API_KEY` and your own organization. The next cell creates a kit holding the built-in **Parallel** web search, which needs no key.

> **Requires:** Tool Gateway (beta) for your organization, in the GCP US region, and a personal access token as `LANGSMITH_API_KEY` (service keys are rejected). **Without it:** the cells print `Skipped:`. Read on for what the trace shows.

In [ ]:
from utils.gateway import create_tool_kit

# personal scope: visible only to you, no admin needed
kit = create_tool_kit(
    "tour-research-kit",
    tools=[{"builtin_server_key": "parallel", "tool_name": "parallel_web_search"}],
)
search_tool_name = kit["tools"][0]["mcp_name"] if kit else None  # the tool's name on the kit's MCP endpoint

print("Tool kit:", f"{kit['name']} · tool {search_tool_name}" if kit else "not created")

Next, swap the agent's own `web_search` for one that calls the kit. It sends your trace headers, so the Tool Gateway's run nests inside the agent's trace, and it records each outcome so section 5 can tell an allowed call from a blocked one. The run cell after it asks one news question; expect an answer and `Tool Gateway calls: ['ok']`.

In [ ]:
from langchain_core.tools import tool

from utils.gateway import ToolCallBlocked, ToolGatewayError, call_kit_tool

kit_calls = []  # one entry per call: "ok", "blocked" or "error"


@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for company news, market data, or general information.

    Args:
        query: Search query to execute.
    """
    try:
        found = call_kit_tool(kit["id"], search_tool_name, {"search_queries": [query], "objective": query})
    except ToolCallBlocked as exc:
        kit_calls.append("blocked")
        return f"Blocked by policy: {exc}"
    except ToolGatewayError as exc:
        kit_calls.append("error")
        return f"Search unavailable: {exc}"
    kit_calls.append("ok")
    return found


kit_agent = build_client_research_agent(model=tour_model, search_tool=web_search)  # same agent, search served by the kit

In [ ]:
kit_question = "What did Microsoft announce about Azure this week? Search at most once."

if kit is None:
    print("Skipped: no tool kit")
else:
    run_id = uuid7()  # lets the cell link straight to this trace
    result = kit_agent.invoke(
        {"messages": [{"role": "user", "content": kit_question}]},
        config={"run_id": run_id, "configurable": {"thread_id": str(uuid4())}},
    )

    print("Answer:", result["messages"][-1].text[:300])
    print("Tool Gateway calls:", kit_calls)
    trace_link(trace_url(run_id), "Agent trace")

Open the trace and expand `web_search`: a child run from the Tool Gateway records the tool, the kit, the identity it acted as and the timing, but never the arguments or results. The same kit endpoint works from Claude Code or Cursor. If `kit_calls` is empty, the agent answered without searching; rerun the cell.

## 5. Block a tool call with a policy
A **tool policy** decides, per caller and tool, whether a call through the Tool Gateway runs. A denied call returns `403` to the agent and is recorded in its trace, so you can cut off a tool for a user or team without redeploying the agent. Policies live in your organization (the one behind `LANGSMITH_API_KEY`) and show up in **Tools** → **Policies**.

The policy below forbids one action, Parallel's web search, for one user: you. Policies name a caller by LangSmith user ID, so set `MY_EMAIL` in the first cell and the policy cell looks your ID up from it.

> **Requires:** tool policies (beta) for your organization and `organization:access-control:manage` there (Organization Admin). **Without it:** leave `create_tool_policy_now = False`; the policy cell prints `Skipped` and the run below is allowed.

In [ ]:
from utils.gateway import builtin_tool_action, create_tool_policy, find_user_id

create_tool_policy_now = False  # an Organization Admin in your org sets this to True
MY_EMAIL = ""  # the email you log in to LangSmith with

block_actions = [builtin_tool_action("parallel", "parallel_web_search")]  # server key + tool name, as in the kit
print("Actions to block:", block_actions)

The next cell creates the policy, or reuses one with the same name.

In [ ]:
if create_tool_policy_now and kit:
    my_user_id = find_user_id(MY_EMAIL)
    if my_user_id is None:
        raise ValueError(f"No member of your organization logs in with {MY_EMAIL!r}; set MY_EMAIL above.")

    tool_policy = create_tool_policy("tour-block-search", my_user_id, block_actions)
    print("Tool policy:", tool_policy["name"] if tool_policy else "not created")
else:
    tool_policy = None
    print("Tool policy: Skipped (an Organization Admin verifies live)")

The policy now appears in **Tools** → **Policies**.

Now rerun the same question through the kit. With the policy in place, expect `Tool Gateway calls: ['blocked']` and a reply that says search was unavailable.

In [ ]:
if kit is None:
    print("Skipped: no tool kit")
else:
    kit_calls.clear()
    run_id = uuid7()  # lets the cell link straight to this trace
    result = kit_agent.invoke(
        {"messages": [{"role": "user", "content": kit_question}]},
        config={"run_id": run_id, "configurable": {"thread_id": str(uuid4())}},
    )

    print("Tool Gateway calls:", kit_calls)
    print("Blocked by policy:", "blocked" in kit_calls)
    print("Agent reply:", result["messages"][-1].text[:300])
    trace_link(trace_url(run_id), "Agent trace")

In the trace, the `web_search` run returns the policy error, and its Tool Gateway child run names `tour-block-search`. In **Organization Settings** → **Audit logs**, set **Operation** to `invoke_mcp_gateway`: each kit call, allowed or denied, is an entry with your identity, the kit and the tool.

> **Requires:** Enterprise plan and Organization Admin or Operator for audit logs. **Without it:** the trace is the record.

## 6. Clean up
The policies and the kit outlive this notebook, so this cell deletes the ones it created.

In [ ]:
from utils.gateway import delete_gateway_policy, delete_tool_kit, delete_tool_policy

deleted = []
for policy in (redaction, spend_cap):
    if policy:
        delete_gateway_policy(policy["id"])
        deleted.append(policy["name"])

if tool_policy:
    delete_tool_policy(tool_policy["id"])
    deleted.append(tool_policy["name"])

if kit:
    delete_tool_kit(kit["id"])
    deleted.append(kit["name"])

print("Deleted:", ", ".join(deleted) or "nothing")

## Recap
| Step | Where it lives |
|---|---|
| Route model calls | `utils/models.py` gateway pair (`LANGSMITH_GATEWAY=true` for ChatAnthropic / Deep Agents apps) · **LLM Gateway** → **Traces** in the gateway key's org |
| Redact secrets | data policy: `guard`, `detect.secrets`, `detect.custom`, `trace.capture_content` · **LLM Gateway** → **Data Policy** |
| Cap spend per team | `spend_cap` + `X-Gateway-*` header condition · **LLM Gateway** → **Cost Controls** |
| Serve tools from a kit | `/v1/managed-tools/kits/{kit_id}/mcp` · **Tools** → **Tool kits** |
| Block a tool | `POST /v1/agent-policies` (`FORBID`, a user, a tool action) · **Tools** → **Policies** |
| Prove it | gateway metadata on model runs, Tool Gateway child runs, audit log `invoke_mcp_gateway` |

**Next:** Module 7 — Deploy

**Docs:** [LLM Gateway](https://docs.langchain.com/langsmith/llm-gateway) · [Direct model access](https://docs.langchain.com/langsmith/llm-gateway-direct-model-access) · [Data policies](https://docs.langchain.com/langsmith/llm-gateway-data-policy) · [Spend policies](https://docs.langchain.com/langsmith/llm-gateway-spend-policies) · [Header-based policies](https://docs.langchain.com/langsmith/llm-gateway-header-policies) · [Gateway traces](https://docs.langchain.com/langsmith/llm-gateway-traces) · [Audit logs](https://docs.langchain.com/langsmith/audit-logs) · [Tool kits in Managed Deep Agents](https://docs.langchain.com/langsmith/managed-deep-agents-search)